In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import random
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder

# --- CONFIG ---
DATA_DIR = Path("/kaggle/input/datasets/mariamhany44/100-words-preprocessed-landmarks-cleaned-no-mask")
BATCH_SIZE = 32
EPOCHS = 150 # OneCycle handles convergence faster
MAX_LR = 6e-4 # Peak LR for OneCycle
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
MAX_FRAMES = 120
PATIENCE = 20
pat_counter = 0
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

# --- DATA LOADING ---
df = pd.read_csv(DATA_DIR / "splits/splits.csv")
df["filepath"] = df["filepath"].str.replace("\\", "/", regex=False)
df["full_path"] = df["filepath"].apply(lambda x: DATA_DIR / "splits" / x)
le = LabelEncoder()
df["label_id"] = le.fit_transform(df["label"])
num_classes = len(le.classes_)
np.save("label_classes.npy", le.classes_)
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

# --- ENHANCED DATASET ---
class ASLDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df
        self.train = train
    def pad_or_truncate(self, x):
        if x.shape[0] > MAX_FRAMES: return x[:MAX_FRAMES]
        elif x.shape[0] < MAX_FRAMES:
            pad = np.zeros((MAX_FRAMES - x.shape[0], x.shape[1]))
            return np.vstack([x, pad])
        return x
    def augment(self, x):
        if self.train:
            # Spatial scaling (simulates distance from camera)
            if random.random() < 0.5:
                scale = np.random.uniform(0.85, 1.15)
                x = x * scale
            # Pointwise jitter
            if random.random() < 0.4:
                x += np.random.normal(0, 0.002, x.shape)
        return x
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = np.load(row["full_path"])
        x = self.pad_or_truncate(x)
        x = self.augment(x)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(row["label_id"])

# --- LOADERS ---
class_counts = train_df["label_id"].value_counts().sort_index().values
class_weights = 1.0 / (class_counts + 1e-6)
sample_weights = train_df["label_id"].map(lambda x: class_weights[x]).values
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(ASLDataset(train_df, True), batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(ASLDataset(val_df, False), batch_size=BATCH_SIZE)
test_loader  = DataLoader(ASLDataset(test_df, False), batch_size=BATCH_SIZE)

# --- ENHANCED MODEL ---
class ResidualBlock(nn.Module):
    def __init__(self, dim, drop_path=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(dim, dim, 3, padding=1, groups=dim), # Depthwise
            nn.Conv1d(dim, dim, 1), # Pointwise
            nn.BatchNorm1d(dim),
            nn.GELU(),
            nn.Dropout(drop_path),
            nn.Conv1d(dim, dim, 3, padding=1),
            nn.BatchNorm1d(dim)
        )
    def forward(self, x): 
        return x + self.conv(x)

class ASLModel(nn.Module):
    def __init__(self, input_dim=438, num_classes=105):
        super().__init__()
        self.input_bn = nn.BatchNorm1d(input_dim)
        self.stem = nn.Sequential(
            nn.Conv1d(input_dim, 256, 7, padding=3),
            nn.BatchNorm1d(256),
            nn.GELU()
        )
        self.res_layers = nn.Sequential(*[ResidualBlock(256) for _ in range(3)])
        self.lstm = nn.LSTM(256, 256, num_layers=2, batch_first=True, bidirectional=True, dropout=0.5)
        
        self.attention = nn.Sequential(
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.input_bn(x)
        x = self.stem(x)
        x = self.res_layers(x)
        x = x.transpose(1, 2) 
        lstm_out, _ = self.lstm(x) 
        
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return self.classifier(context)

model = ASLModel(num_classes=num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=0.05)

# OneCycleLR is better for fine-tuning
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR, steps_per_epoch=len(train_loader), epochs=EPOCHS
)

# --- TRAINING LOOP ---
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct, total = 0, 0, 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        
        # Probabilistic Mixup
        if random.random() < 0.5:
            lam = np.random.beta(0.2, 0.2)
            perm = torch.randperm(x.size(0)).to(DEVICE)
            outputs = model(lam * x + (1 - lam) * x[perm])
            loss = lam * criterion(outputs, y) + (1 - lam) * criterion(outputs, y[perm])
            train_correct += (lam * (outputs.argmax(1) == y).sum().item() + (1 - lam) * (outputs.argmax(1) == y[perm]).sum().item())
        else:
            outputs = model(x)
            loss = criterion(outputs, y)
            train_correct += (outputs.argmax(1) == y).sum().item()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        train_loss += loss.item() * y.size(0)
        total += y.size(0)

    # Validation
    model.eval()
    val_loss, val_correct, total_val = 0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            outputs = model(x)
            v_loss = criterion(outputs, y)
            val_loss += v_loss.item() * y.size(0)
            val_correct += (outputs.argmax(1) == y).sum().item()
            total_val += y.size(0)
    
    t_loss, t_acc = train_loss/total, train_correct/total
    v_loss, v_acc = val_loss/total_val, val_correct/total_val
    for k, v in zip(history.keys(), [t_loss, v_loss, t_acc, v_acc]): history[k].append(v)

    print(f"Epoch {epoch+1:03d} | T-Loss: {t_loss:.3f} | T-Acc: {t_acc:.3f} | V-Loss: {v_loss:.3f} | V-Acc: {v_acc:.3f}")

    if v_acc > best_val:
        best_val = v_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "label_classes": le.classes_,
            "config": {
                "input_dim": 438,
                "num_classes": num_classes,
                "MAX_FRAMES": MAX_FRAMES
            },
            "best_val_acc": best_val
        }, "best_refined_model.pth")
    else:
        pat_counter += 1
        if pat_counter >= PATIENCE:
            print(f"⛔ Early stopping triggered at epoch {epoch+1}")
            break

# --- PLOTTING ---
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.plot(history["train_loss"], label="Train"); plt.plot(history["val_loss"], label="Val"); plt.title("Refined Loss"); plt.legend()
plt.subplot(1, 2, 2); plt.plot(history["train_acc"], label="Train"); plt.plot(history["val_acc"], label="Val"); plt.title("Refined Accuracy"); plt.legend()
plt.savefig("training_curves.png")
plt.show()

# --- LOAD BEST MODEL CHECKPOINT ---
# Use weights_only=False since the checkpoint has extra objects (optimizer, config, labels)
checkpoint = torch.load("best_refined_model.pth", map_location=DEVICE, weights_only=False)

# Restore model and optimizer state
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

# Restore label encoder classes
le.classes_ = checkpoint["label_classes"]
num_classes = len(le.classes_)

# --- EVALUATE ON TEST SET ---
model.eval()
preds, labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(DEVICE))
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(y.numpy())

# Convert predictions back to original labels
pred_labels = le.inverse_transform(preds)
true_labels = le.inverse_transform(labels)

print(f"\nFinal Test Accuracy: {np.mean(np.array(preds) == np.array(labels)):.4f}")

Epoch 001 | T-Loss: 4.642 | T-Acc: 0.026 | V-Loss: 4.563 | V-Acc: 0.050
Epoch 002 | T-Loss: 4.460 | T-Acc: 0.058 | V-Loss: 4.308 | V-Acc: 0.115


KeyboardInterrupt: 